<a href="https://colab.research.google.com/github/cyberviser/Hancock/blob/main/Hancock_Universal_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyberviser/Hancock/blob/main/Hancock_Universal_Finetune.ipynb)

# 🔐 Hancock Universal Fine-Tuning — CyberViser
**Mistral 7B → Cybersecurity specialist via LoRA**

Works on: **Google Colab** (free T4) · **Kaggle** (free T4, 30h/week) · **RunPod/Vast** · **Oracle Cloud**

| Step | Time | Notes |
|------|------|-------|
| Install deps | ~3 min | Unsloth + TRL |
| Load 4-bit model | ~2 min | 7B params, 4GB VRAM |
| Train (300 steps) | ~25 min | v3 dataset (5,670 samples) |
| Export GGUF Q4 | ~5 min | Ready for Ollama |

**Setup:**
- **Colab:** Runtime → Change runtime type → T4 GPU
- **Kaggle:** Settings → Accelerator → GPU T4 x2 · Internet → On
- **Optional:** Add `HF_TOKEN` secret to push model to HuggingFace Hub

In [ ]:
import os
print(f"📂 Current Directory: {os.getcwd()}")
print("📄 Files found:")
!ls -F

if os.path.exists('hancock-adapter-v3'):
    print("\n⚠️ Adapter folder exists. Checking contents...")
    !ls -F hancock-adapter-v3

In [377]:
import psutil

# Check for running python processes executing the fine-tuning script
training_running = False
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        cmdline = proc.info['cmdline']
        if cmdline and 'hancock_finetune_v3.py' in ' '.join(cmdline):
            print(f"🔄 Training is STILL RUNNING (PID: {proc.info['pid']})")
            training_running = True
            break
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

if not training_running:
    print("✅ No training script found running. It has likely FINISHED (or failed).")
    print("   Please check the bottom of the Step 5 output for results.")

✅ No training script found running. It has likely FINISHED (or failed).
   Please check the bottom of the Step 5 output for results.


In [448]:
import os
import datetime

print(f"🔍 Diagnostics at {datetime.datetime.now().strftime('%H:%M:%S')}...")
print(f"📂 Current Working Directory: {os.getcwd()}")

# List all directories to find where the model went
print("\n📂 Subdirectories found:")
found_dirs = [d for d in os.listdir('.') if os.path.isdir(d) and not d.startswith('.')]
if not found_dirs:
    print("   (No subdirectories found)")
else:
    for d in found_dirs:
        print(f"   - {d}")

# Check for GGUF files
print("\n📄 GGUF files found:")
ggufs = [f for f in os.listdir('.') if f.endswith('.gguf')]
if not ggufs:
    print("   (No .gguf files found)")
else:
    for f in ggufs:
        print(f"   - {f}")

# Specific check
if 'hancock-adapter-v3' in found_dirs:
    print("\n✅ Standard adapter folder 'hancock-adapter-v3' found!")
elif 'hancock-cpu-adapter' in found_dirs:
    print("\n⚠️ Found 'hancock-cpu-adapter' instead. This might be a CPU-optimized run.")
    print("   You may need to update Step 6 to use this folder name.")
else:
    print("\n❌ No obvious adapter folder found. Step 5 likely failed or is still running.")

🔍 Diagnostics at 22:46:10...
📂 Current Working Directory: /content/Hancock

📂 Subdirectories found:
   - unsloth_compiled_cache
   - hancock-adapter-v3
   - clients
   - tests
   - hancock-cpu-adapter
   - collectors
   - huggingface_tokenizers_cache
   - formatter
   - data
   - docs

📄 GGUF files found:
   (No .gguf files found)

✅ Standard adapter folder 'hancock-adapter-v3' found!


In [ ]:
print('Contents of /content:')
!ls -F /content

print('\nContents of /content/Hancock:')
!ls -F /content/Hancock

In [150]:
# @title 1️⃣  Detect Environment
import os, platform

ENV = 'Unknown'
if 'COLAB_GPU' in os.environ or os.path.exists('/content'):
    ENV = 'Colab'
    WORK_DIR = '/content'
elif os.path.exists('/kaggle'):
    ENV = 'Kaggle'
    WORK_DIR = '/kaggle/working'
else:
    ENV = 'Cloud/Local'
    WORK_DIR = os.getcwd()

print(f'💻 Environment: {ENV}')
print(f'📁 Work dir: {WORK_DIR}')
os.chdir(WORK_DIR)

💻 Environment: Colab
📁 Work dir: /content


In [151]:
print(f'Contents of {WORK_DIR}:')
!ls -F {WORK_DIR}

Contents of /content:
Hancock/  sample_data/


In [152]:
# @title 2️⃣  Install Dependencies (~3 min)
import subprocess, sys
import importlib.metadata

# Check if ENV is defined, otherwise default to Colab install
target_env = 'Colab'
if 'ENV' in globals():
    target_env = ENV

# Unsloth install varies by environment
if target_env == 'Kaggle':
    !pip install 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git' -q
else:
    !pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q

!pip install -q 'trl>=0.8.6' 'transformers>=4.40' 'datasets>=2.18' peft huggingface_hub accelerate bitsandbytes

for pkg in ['unsloth','trl','transformers','datasets','peft']:
    try:
        v = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        v = 'not installed'
    print(f'  {pkg}: {v}')
print('\n✅ Dependencies installed')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  unsloth: 2026.2.1
  trl: 0.22.0
  transformers: 4.56.0
  datasets: 4.3.0
  peft: 0.18.1

✅ Dependencies installed


In [153]:
# @title 3️⃣  Clone Hancock & Check GPU
import torch
import os

# Ensure WORK_DIR is defined
if 'WORK_DIR' not in globals():
    WORK_DIR = '/content' if os.path.exists('/content') else os.getcwd()

# Clone repo
if not os.path.exists(os.path.join(WORK_DIR, 'Hancock')):
    !git clone https://github.com/cyberviser/Hancock.git {WORK_DIR}/Hancock
os.chdir(os.path.join(WORK_DIR, 'Hancock'))

# GPU check
assert torch.cuda.is_available(), '❌ Enable GPU runtime first!'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
print(f'Repo: {os.getcwd()}')
print('✅ Ready')

GPU: Tesla T4 | VRAM: 15.6 GB
Repo: /content/Hancock
✅ Ready


In [154]:
# @title 4️⃣  Configure Training
# Adjust these settings before running:

TRAINING_STEPS = 300        # More steps = better quality, longer time (~5 min per 100 steps on T4)
EXPORT_GGUF = True          # Export GGUF for Ollama deployment
PUSH_TO_HUB = False         # Push to HuggingFace Hub (requires HF_TOKEN)
LORA_RANK = 0               # 0 = auto-detect based on VRAM (16=T4, 32=24GB, 64=40GB+)

print(f'Steps: {TRAINING_STEPS}')
print(f'GGUF export: {EXPORT_GGUF}')
print(f'Push to Hub: {PUSH_TO_HUB}')
print(f'LoRA rank: {"auto" if LORA_RANK == 0 else LORA_RANK}')

Steps: 300
GGUF export: True
Push to Hub: False
LoRA rank: auto


In [155]:
# @title 5️⃣  Run Fine-Tuning (~25 min on T4)
import os
import subprocess
import sys

# Robust directory switching
possible_dirs = ['/content/Hancock', '/kaggle/working/Hancock', os.path.join(os.getcwd(), 'Hancock')]
for d in possible_dirs:
    if os.path.exists(d):
        os.chdir(d)
        break

print(f"📂 Working in: {os.getcwd()}")
print("🚀 Starting Fine-Tuning... Please wait.")

# Ensure variables exist
if 'TRAINING_STEPS' not in globals(): TRAINING_STEPS = 300
if 'EXPORT_GGUF' not in globals(): EXPORT_GGUF = True
if 'PUSH_TO_HUB' not in globals(): PUSH_TO_HUB = False
if 'LORA_RANK' not in globals(): LORA_RANK = 0

script_name = 'hancock_finetune_v3.py'
if not os.path.exists(script_name):
    print(f"❌ Error: {script_name} not found in {os.getcwd()}")
    print("Please re-run Step 3 to clone the repository.")
else:
    # Prepare command args
    cmd_args = [sys.executable, script_name, '--steps', str(TRAINING_STEPS)]
    if EXPORT_GGUF:
        cmd_args.append('--export-gguf')
    if PUSH_TO_HUB:
        cmd_args.append('--push-to-hub')
    if LORA_RANK > 0:
        cmd_args.extend(['--lora-r', str(LORA_RANK)])

    print(f"Running: {' '.join(cmd_args)}\n")

    # Prepare environment with log suppression
    env = os.environ.copy()
    env['TF_CPP_MIN_LOG_LEVEL'] = '3'
    env['PYTHONUNBUFFERED'] = '1'

    # Run with direct stream merging
    process = subprocess.Popen(
        cmd_args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, # Merge stderr into stdout
        text=True,
        bufsize=1,
        universal_newlines=True,
        env=env
    )

    # Stream output
    for line in process.stdout:
        print(line, end='') # Use print to handle newlines correctly

    process.wait()
    if process.returncode != 0:
        print(f"\n❌ Training failed with exit code {process.returncode}")

📂 Working in: /content/Hancock
🚀 Starting Fine-Tuning... Please wait.
Running: /usr/bin/python3 hancock_finetune_v3.py --steps 300 --export-gguf


[v3] Hancock Fine-Tune v3 — CyberViser
     Platform : Linux (Colab)
     GPU      : Tesla T4
     VRAM     : 15.6 GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
E0000 00:00:1772486978.245361  159129 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772486978.280817  159129 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772486978.338405  159129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772486978.338642  159129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid

In [156]:
# @title 6️⃣  Test the Fine-Tuned Model
import torch
from unsloth import FastLanguageModel
from pathlib import Path
import os

# Ensure we are in the repo or can find the adapter
adapter_name = 'hancock-adapter-v3'
found = False

# Search likely locations
possible_paths = [
    Path(adapter_name),
    Path('/content/Hancock') / adapter_name,
    Path('/kaggle/working/Hancock') / adapter_name,
    Path(os.getcwd()) / adapter_name
]

for p in possible_paths:
    if p.exists():
        adapter_name = str(p)
        found = True
        break

if not found:
    print(f"❌ Error: Adapter folder '{adapter_name}' not found.")
    print("Did Step 5 finish successfully? Check the output of the training cell.")
else:
    print(f"✅ Loading model from: {adapter_name}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_name,
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    messages = [
        {'role': 'system', 'content': 'You are Hancock, an elite cybersecurity AI by CyberViser.'},
        {'role': 'user', 'content': 'Explain CVE-2021-44228 Log4Shell and provide a Splunk SPL query to detect exploitation attempts.'},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')

    print("\n🤔 Hancock is thinking...\n")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=512,
            use_cache=True, temperature=0.7, do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print('🛡️ Hancock says:')
    print('=' * 60)
    print(response)

❌ Error: Adapter folder 'hancock-adapter-v3' not found.
Did Step 5 finish successfully? Check the output of the training cell.


In [157]:
# @title 7️⃣  Download Model Files
import os
from pathlib import Path

print('\n📦 Available model files:\n')

# LoRA adapter
adapter_dir = Path('hancock-adapter-v3')
if adapter_dir.exists():
    size = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
    print(f'  📁 hancock-adapter-v3/ ({size:.0f} MB) — LoRA adapter')

# GGUF
for gguf in Path('.').glob('*.gguf'):
    size = gguf.stat().st_size / 1e6
    print(f'  📁 {gguf.name} ({size:.0f} MB) — GGUF for Ollama')

print('\n⬇️  Download options:')
if ENV == 'Colab':
    print('  Option 1: Run the cell below to download via browser')
    print('  Option 2: Mount Google Drive and copy there')
elif ENV == 'Kaggle':
    print('  Option 1: Output tab → download files from /kaggle/working/Hancock/')
    print('  Option 2: Push to HuggingFace Hub (re-run with PUSH_TO_HUB=True)')
else:
    print('  Files are saved in the current directory')


📦 Available model files:


⬇️  Download options:
  Option 1: Run the cell below to download via browser
  Option 2: Mount Google Drive and copy there


In [158]:
# @title 8️⃣  Download GGUF (Colab only)
# Skip this cell on Kaggle — use Output tab instead
import os

try:
    from google.colab import files
    for f in os.listdir('.'):
        if f.endswith('.gguf'):
            print(f'Downloading {f}...')
            files.download(f)
            break
    else:
        print('No GGUF file found. Run with EXPORT_GGUF=True')
except ImportError:
    print('Not running on Colab — download manually from the Output tab (Kaggle) or local filesystem')

No GGUF file found. Run with EXPORT_GGUF=True


---
## 🚀 Deploy with Ollama (after download)

```bash
# On your local machine:
mkdir -p ~/cyberviser/models && mv hancock-v3-Q4_K_M.gguf ~/cyberviser/models/
cd ~/cyberviser/Hancock

# Update Modelfile to point to GGUF:
# FROM ./models/hancock-v3-Q4_K_M.gguf

ollama create hancock-finetuned -f Modelfile.hancock-finetuned
ollama run hancock-finetuned
```

---
© 2026 CyberViser · [cyberviser.netlify.app](https://cyberviser.netlify.app)

# Task
Display the `top_features` DataFrame to show the coefficients and relative importance of the features from the linear classification model, and then generate a bar chart to visualize their `Relative_Importance (%)`. Next, display the `xgb_importance_df` DataFrame to show the feature importance rankings from the XGBoost model, and generate a corresponding bar chart to visualize these importances for comparison. Finally, summarize the key findings regarding which features are most important for classification across both models.

## Display Linear Model Feature Importance

### Subtask:
Display the dataframe containing feature importance metrics for the linear model.


**Reasoning**:
The user wants to inspect the `top_features` dataframe which contains the feature importance metrics. I will display this dataframe.



In [ ]:
print("Linear Model Feature Importance:")
display(top_features)